In [ ]:
# ============================================================================================
# SEL 1 - START.  Jalankan sel ini SAJA (bukan Run All), lalu klik tautan yang muncul di bawah.
#
#  - Menyalakan Neo4j lokal di latar belakang (hanya bisa diakses dari komputer ini).
#  - Mengisi graf dari CSV kalau masih kosong.
#  - Menampilkan tautan Neo4j Browser + contoh query untuk dicoba.
#  - JANGAN tutup / restart kernel selama menjelajah: Neo4j ikut berhenti kalau kernel dimatikan.
#  - Selesai menjelajah? Jalankan SEL 2 untuk mematikan Neo4j.
# ============================================================================================
import sys
from pathlib import Path

from IPython.display import Markdown, display

ROOT = Path.cwd() if (Path.cwd() / "scripts").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "scripts"))
import neo4j_lokal

if neo4j_lokal.nyalakan():
    n = neo4j_lokal.jumlah_user()
    if n is None or n == -1:
        print("\nLogin ke Neo4j gagal: periksa NEO4J_PASSWORD di file .env (harus sama dengan password Neo4j).")
    else:
        if n == 0:
            print("\nGraf masih kosong. Mengisi dari CSV (scripts/03_sync_to_neo4j.py) ...")
            neo4j_lokal.jalankan_script("03_sync_to_neo4j.py")
            n = neo4j_lokal.jumlah_user()

        # contoh: user dengan jumlah teman di persentil ke-75, dan beberapa nama hobi yang ada
        derajat = neo4j_lokal.baca("MATCH (u:User) WITH u, COUNT { (u)-[:FRIEND]->() } AS d WHERE d > 0 "
                                   "RETURN u.user_id AS id ORDER BY d DESC, id")
        contoh = derajat[int(len(derajat) * 0.25)]["id"]
        hobi = neo4j_lokal.baca("MATCH (:User)-[r:HAS_HOBBY]->(h:Hobby) RETURN h.name AS nama, count(r) AS n "
                                "ORDER BY n DESC, nama")
        nama_hobi = hobi[len(hobi) // 2]["nama"]      # hobi berpopularitas median: tidak terlalu ramai, tidak terlalu sepi

        print(f"\nNeo4j siap dan graf terisi ({n:,} user).")
        print("\n1) Buka Neo4j Browser di tab browser:  http://localhost:7474")
        print("2) Di dialog 'Connect to instance' (sudah terisi, tinggal password):")
        print("     Protocol        : neo4j://   (kalau gagal, ganti ke bolt://)")
        print("     Connection URL  : localhost:7687")
        print("     Database user   : neo4j")
        print(f"     Password        : isi dengan NEO4J_PASSWORD dari file  {ROOT / '.env'}")
        print("     (biarkan 'Connect with SSO' mati, lalu klik Connect)")
        print("3) Tempel salah satu query di bawah ke kotak query (baris paling atas), tekan Ctrl+Enter, lalu geser / klik / dobel-klik node.\n")
        queries = [
            ("Jaringan pertemanan (100 relasi pertama)",
             "MATCH (u:User)-[f:FRIEND]->(v:User) RETURN u, f, v LIMIT 100"),
            (f"Teman dari teman user {contoh} (siapa yang bisa direkomendasikan)",
             f"MATCH (me:User {{user_id: {contoh}}})-[:FRIEND]->(f:User)-[:FRIEND]->(c:User) "
             f"WHERE c <> me AND NOT (me)-[:FRIEND]-(c) RETURN me, f, c LIMIT 120"),
            ("Hub terbesar dan teman-temannya",
             "MATCH (h:User) WITH h, COUNT { (h)-[:FRIEND]-() } AS d ORDER BY d DESC LIMIT 1 "
             "MATCH p = (h)-[:FRIEND]-(v:User) RETURN p LIMIT 150"),
            (f"Siapa saja yang berhobi '{nama_hobi}'",
             f"MATCH (u:User)-[h:HAS_HOBBY]->(b:Hobby {{name: '{nama_hobi}'}}) RETURN u, h, b LIMIT 120"),
            ("Model graf (node dan relasi apa saja)", "CALL db.schema.visualization()"),
        ]
        for judul, q in queries:
            print(f"-- {judul}\n{q}\n")
        display(Markdown("### [Buka Neo4j Browser (http://localhost:7474)](http://localhost:7474)"))


In [2]:
# ============================================================================================
# SEL 2 - STOP.  Jalankan kalau sudah selesai menjelajah: mematikan Neo4j.
#  (Dihentikan paksa; aman, Neo4j memulihkan diri saat dinyalakan lagi.)
# ============================================================================================
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "scripts").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "scripts"))
import neo4j_lokal

neo4j_lokal.matikan()


[OK] Neo4j dimatikan.
